# 10 Beta Diversity and Age

We test whether fungal community composition is associated with age in years (`age_years_sample`). The primary analysis uses 16-mer Bray–Curtis distances, and the sensitivity check uses whole-ASV Bray–Curtis distances.

1. Summarize age and missing values among the 150 samples from notebook 07.
2. Subset both distance matrices to the same samples with a usable age value.
3. Test the association with age as a continuous variable and compare the results across feature representations.

In [15]:
import pandas as pd
import qiime2 as q2
from qiime2 import Visualization
from skbio import DistanceMatrix

In [9]:
kmer16_distance_path = "../data/processed_ignored/07_beta_diversity/kmer16_bray_curtis_distance.qza"
asv_distance_path = "../data/processed_ignored/07_beta_diversity/asv_bray_curtis_distance.qza"
metadata_path = "../data/processed_ignored/06_diversity/Final_metadata_with_GHSL.tsv"

out_dir = "../data/processed_ignored/10_beta_diversity_age"
vis_dir = "../data/processed/10_beta_diversity_age"

age_metadata_path = f"{out_dir}/age_metadata.tsv"
kmer16_age_distance_path = f"{out_dir}/kmer16_bray_curtis_age.qza"
asv_age_distance_path = f"{out_dir}/asv_bray_curtis_age.qza"

kmer16_age_permanova_path = f"{vis_dir}/kmer16_bray_curtis_age_permanova.qzv"
asv_age_permanova_path = f"{vis_dir}/asv_bray_curtis_age_permanova.qzv"

! mkdir -p "$out_dir" "$vis_dir"

## 1. Summarize age

Summarize the distribution of age in years among the samples in the full distance matrix. Count missing or non-numeric age values.

In [3]:
metadata = q2.Metadata.load(metadata_path).to_dataframe()
distance_matrix = q2.Artifact.load(kmer16_distance_path).view(DistanceMatrix)

age = pd.to_numeric(
    metadata.reindex(list(distance_matrix.ids))["age_years_sample"],
    errors="coerce",
)

print("Missing or non-numeric ages:", age.isna().sum())
age.describe()

Missing or non-numeric ages: 5


count    145.000000
mean      48.779310
std       18.677998
min        2.000000
25%       36.000000
50%       53.000000
75%       63.000000
max       82.000000
Name: age_years_sample, dtype: float64

## 2. Prepare the age metadata and distance matrices

Keep the 145 samples with a numeric age and save their ages in years. Subset both Bray–Curtis distance matrices to the same sample IDs.

In [5]:
age_metadata = age.dropna().to_frame()
age_sample_ids = age_metadata.index.tolist()
q2.Metadata(age_metadata).save(age_metadata_path)

kmer16_age = distance_matrix.filter(age_sample_ids)
q2.Artifact.import_data("DistanceMatrix", kmer16_age).save(kmer16_age_distance_path)

asv_distance = q2.Artifact.load(asv_distance_path).view(DistanceMatrix)
asv_age = asv_distance.filter(age_sample_ids)
q2.Artifact.import_data("DistanceMatrix", asv_age).save(asv_age_distance_path)

'../data/processed_ignored/10_beta_diversity_age/asv_bray_curtis_age.qza'

## 3. Test for an association with age

### 3.1 PERMANOVA on 16-mer Bray–Curtis distances

Use ADONIS to test whether fungal community composition is associated with age in years. Include age as a continuous predictor and use 999 permutations across the 145 samples with a numeric age.

In [10]:
! qiime diversity adonis \
    --i-distance-matrix "$kmer16_age_distance_path" \
    --m-metadata-file "$age_metadata_path" \
    --p-formula "age_years_sample" \
    --o-visualization "$kmer16_age_permanova_path"

Saved Visualization to: ../data/processed/10_beta_diversity_age/kmer16_bray_curtis_age_permanova.qzv


In [ ]:
Visualization.load(kmer16_age_permanova_path)

### 3.2 PERMANOVA on whole-ASV Bray–Curtis distances

Repeat the ADONIS test using whole-ASV Bray–Curtis distances. Use the same 145 samples, age in years as a continuous predictor, and 999 permutations to see whether the result depends on the feature representation.

In [13]:
! qiime diversity adonis \
    --i-distance-matrix "$asv_age_distance_path" \
    --m-metadata-file "$age_metadata_path" \
    --p-formula "age_years_sample" \
    --o-visualization "$asv_age_permanova_path"

Saved Visualization to: ../data/processed/10_beta_diversity_age/asv_bray_curtis_age_permanova.qzv


In [ ]:
Visualization.load(asv_age_permanova_path)

### 3.3 Compare the age PERMANOVA results

| Bray–Curtis feature representation | Samples | R² | Pseudo-F | p-value |
|---|---:|---:|---:|---:|
| 16-mers | 145 | 0.0153 | 2.223 | **0.034** |
| Whole ASVs | 145 | 0.0112 | 1.613 | 0.086 |

**Results**

At α = 0.05, the primary 16-mer analysis detected an association between age and fungal community composition (p = 0.034). The whole-ASV sensitivity check did not reach the same threshold (p = 0.086). Age accounted for a small proportion of variation in both analyses, about 1.5% for 16-mers and 1.1% for whole ASVs ($R^2 = \mathrm{SS}_{\mathrm{age}} / \mathrm{SS}_{\mathrm{total}}$, where SS means sum of squares; multiply $R^2$ by 100 to get the percentage).

## 4. Discussion

Age explained only a small share of the variation in fungal community composition in both analyses. The 16-mer analysis detected an association at the 0.05 threshold, while the whole-ASV analysis did not. Since both tests used the same 145 samples and Bray–Curtis distances, the statistical conclusion was sensitive to how the ITS1 sequences were represented.

The age-colored PCoA plots in notebook 07 showed no clear age gradient along the first two axes. The 16-mer test used the full distance matrix, so a small association can be detectable even when it is hard to see in the two-dimensional plot.